# 03 -- Training

BERT and CNN2D training loops, plus the fake-review detector's four training iterations (results comparison). Standalone-runnable: real training happens live if you run this file alone (takes real GPU/CPU time).

In [ ]:
import os
import sys
from pathlib import Path


def _find_project_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / "backend" / "app").is_dir() and (candidate / "data").is_dir():
            return candidate
        alt = candidate / "Olist_Marketplace_Platform"
        if (alt / "backend" / "app").is_dir() and (alt / "data").is_dir():
            return alt
    raise RuntimeError("Could not locate the project root above this notebook.")


PROJECT_ROOT = _find_project_root(Path.cwd())
os.chdir(PROJECT_ROOT)
sys.path.insert(0, str(PROJECT_ROOT / "backend"))
print("Project root:", PROJECT_ROOT)


In [ ]:
# Standalone setup: rebuilds the exact same deduplicated train/val split this
# notebook's own §6A produces (dedup-before-split, seed 42), and the CNN2D
# tokenizer + model instance from §6B -- via this project's real, shared
# functions (backend/app/ml) instead of re-typing preprocessing.ipynb's logic
# here. Training itself below is 100% the original notebook's own code.
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer

from app.ml.preprocessing import build_sentiment_dataframe, remove_duplicate_reviews, split_sentiment_dataset, SimpleVocabTokenizer
from app.ml.datasets import encode_texts_for_cnn
from app.ml.models import CNN2DReviewSentiment, CNN_MAX_WORDS, CNN_MAX_LEN
from app.ml.utils import set_seed, get_device

SEED = 42
set_seed(SEED)
device = get_device()

reviews = pd.read_csv("data/interim/reviews_translated.csv")
sent_df = build_sentiment_dataframe(reviews)
deduped, _ = remove_duplicate_reviews(sent_df)
split = split_sentiment_dataset(deduped, seed=SEED)
X_train, y_train = split.train["text"], split.train["label"]
X_val, y_val = split.val["text"], split.val["label"]
X_test, y_test = split.test["text"], split.test["label"]  # built by §6B's training cell too (unused until eval)

BERT_CHECKPOINT = "LiYuan/amazon-review-sentiment-analysis"
tokenizer_bert = AutoTokenizer.from_pretrained(BERT_CHECKPOINT)
BERT_MAX_LEN = 128


class ReviewSentimentDataset(Dataset):
    def __init__(self, texts, labels):
        self.texts, self.labels = list(texts), list(labels)

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        return self.texts[idx], int(self.labels[idx])


def make_collate_fn(tokenizer, max_len=BERT_MAX_LEN):
    def collate_fn(batch):
        texts, labels = zip(*batch)
        encodings = tokenizer(list(texts), max_length=max_len, padding=True, truncation=True, return_tensors="pt")
        encodings["labels"] = torch.tensor(labels, dtype=torch.long)
        return encodings
    return collate_fn


bert_collate_fn = make_collate_fn(tokenizer_bert)
train_loader_bert = DataLoader(ReviewSentimentDataset(X_train.astype(str).tolist(), y_train.values), batch_size=8, shuffle=True, collate_fn=bert_collate_fn)
val_loader_bert = DataLoader(ReviewSentimentDataset(X_val.astype(str).tolist(), y_val.values), batch_size=8, shuffle=False, collate_fn=bert_collate_fn)

cnn_tokenizer = SimpleVocabTokenizer(num_words=CNN_MAX_WORDS)
cnn_tokenizer.fit_on_texts(X_train)
X_tr_seq = encode_texts_for_cnn(X_train, cnn_tokenizer, max_len=CNN_MAX_LEN)
X_vl_seq = encode_texts_for_cnn(X_val, cnn_tokenizer, max_len=CNN_MAX_LEN)
X_te_seq = encode_texts_for_cnn(X_test, cnn_tokenizer, max_len=CNN_MAX_LEN)
model_cnn2d = CNN2DReviewSentiment().to(device)

print(f"Prepared {len(X_train):,} train / {len(X_val):,} val examples. BERT_CHECKPOINT={BERT_CHECKPOINT}")


In [ ]:
import time
import copy
import os
import gc
import torch
from transformers import AutoModelForSequenceClassification, get_linear_schedule_with_warmup

# 1. Clear memory before starting
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

# 2. Check and assign Hardware Acceleration
print("====== [1] Checking Hardware Acceleration ======")
if torch.cuda.is_available():
    device = torch.device("cuda")
    print(f"✔ GPU is available: {torch.cuda.get_device_name(0)}")
else:
    device = torch.device("cpu")
    print("⚠ No GPU found. Running on CPU (training will be slow).")

print("\n====== [2] Building BERT Model: Fine-Tuning ======")

# FIX: this cell previously used DistilBertForSequenceClassification and a silent
# fallback to a THIRD, different checkpoint name ("distilbert-base-uncased") if
# BERT_CHECKPOINT wasn't already defined -- both inconsistent with the checkpoint
# actually fixed in the tokenizer cell above. AutoModelForSequenceClassification
# loads whatever architecture BERT_CHECKPOINT actually is; no more silent fallback
# to an untracked, different checkpoint.
#
# SECOND FIX (found by actually running this cell): LiYuan/amazon-review-sentiment-
# analysis ships a 5-label classification head (1-5 star rating), not 2. Loading it
# with num_labels=2 and no further argument raises a state_dict size-mismatch error
# (shape [5,768] vs [2,768]) instead of silently reinitializing the head.
# ignore_mismatched_sizes=True tells transformers to keep the pretrained encoder
# weights and reinitialize ONLY the mismatched classification head for our binary
# task -- matches backend/app/ml/models.py::create_bert_model exactly.
# THIRD FIX (found by actually running this cell on GPU): loading the checkpoint's
# pytorch_model.bin trips a torch<2.6 security guard in recent `transformers`
# (CVE-2025-32434) regardless of weights_only=True. The checkpoint also publishes a
# model.safetensors (a newer, safe-by-construction format) -- forcing it sidesteps
# the restriction entirely, exactly as the error message itself suggests, without
# needing to bump torch past this project's pinned <2.6 upper bound.
model_bert = AutoModelForSequenceClassification.from_pretrained(
    BERT_CHECKPOINT, num_labels=2, ignore_mismatched_sizes=True, use_safetensors=True,
)
model_bert.config.id2label = {0: "Negative", 1: "Positive"}
model_bert.config.label2id = {"Negative": 0, "Positive": 1}
model_bert.to(device)

# 4. Hyperparameters
EPOCHS = 3
LEARNING_RATE = 2e-5
WEIGHT_DECAY = 0.01
EARLY_STOPPING_PATIENCE = 2

total_train_steps = len(train_loader_bert) * EPOCHS
warmup_steps = int(0.1 * total_train_steps)

# 5. Optimizer & Scheduler
optimizer_bert = torch.optim.AdamW(model_bert.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
scheduler_bert = get_linear_schedule_with_warmup(
    optimizer_bert, num_warmup_steps=warmup_steps, num_training_steps=total_train_steps
)

# 6. Output directory
os.makedirs("weights", exist_ok=True)

# 7. Native PyTorch Training & Evaluation Function
def run_bert_epoch(model, loader, optimizer=None, scheduler=None):
    '''Runs one full pass over `loader`. Trains if `optimizer` is given, else evaluates.'''
    is_train = optimizer is not None
    model.train() if is_train else model.eval()

    total_loss, correct, total = 0.0, 0, 0
    
    # تعطيل الـ Gradients تماماً عند التقييم لتوفير الذاكرة
    with torch.set_grad_enabled(is_train):
        for batch in loader:
            # نقل البيانات للـ GPU (cuda)
            batch = {k: v.to(device) for k, v in batch.items()}
            
            # Forward pass
            outputs = model(**batch)
            loss = outputs.loss

            if is_train:
                optimizer.zero_grad()
                loss.backward()
                # حماية الـ Gradients من الانفجار (Gradient Clipping)
                torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
                optimizer.step()
                scheduler.step()

            # حساب الدقة والخسارة
            preds = torch.argmax(outputs.logits, dim=1)
            correct += (preds == batch["labels"]).sum().item()
            total += batch["labels"].size(0)
            total_loss += loss.item() * batch["labels"].size(0)

    return total_loss / total, correct / total


print("\nInitiating Transformer Fine-Tuning loop...")
start_time = time.time()

history_bert = {"accuracy": [], "loss": [], "val_accuracy": [], "val_loss": []}
best_val_acc = -1.0
best_state_dict = None
epochs_without_improvement = 0

# 8. Main Loop
for epoch in range(1, EPOCHS + 1):
    train_loss, train_acc = run_bert_epoch(model_bert, train_loader_bert, optimizer_bert, scheduler_bert)
    val_loss, val_acc = run_bert_epoch(model_bert, val_loader_bert)

    history_bert["loss"].append(train_loss)
    history_bert["accuracy"].append(train_acc)
    history_bert["val_loss"].append(val_loss)
    history_bert["val_accuracy"].append(val_acc)

    print(f"Epoch {epoch}/{EPOCHS} — loss: {train_loss:.4f} - accuracy: {train_acc:.4f} "
          f"- val_loss: {val_loss:.4f} - val_accuracy: {val_acc:.4f}")

    # Save Checkpoint
    ckpt_path = f"weights/bert_review_sentiment_epoch_{epoch:02d}.pt"
    torch.save(model_bert.state_dict(), ckpt_path)
    print(f"  -> checkpoint saved: {ckpt_path}")

    # Early Stopping
    if val_acc > best_val_acc:
        best_val_acc = val_acc
        best_state_dict = copy.deepcopy(model_bert.state_dict())
        epochs_without_improvement = 0
    else:
        epochs_without_improvement += 1
        if epochs_without_improvement >= EARLY_STOPPING_PATIENCE:
            print(f"  -> Early stopping triggered (no val_accuracy improvement for {EARLY_STOPPING_PATIENCE} epochs)")
            break

if best_state_dict is not None:
    model_bert.load_state_dict(best_state_dict)
    print(f"\n✔ Restored best weights (val_accuracy = {best_val_acc:.4f})")

bert_train_time = time.time() - start_time
print(f"\n✔ BERT model training complete! Executed in: {bert_train_time:.2f} seconds.")

In [57]:
from sklearn.utils.class_weight import compute_class_weight
from torch.utils.data import TensorDataset

# Reuse the same SEED fixed in §6 for reproducibility
set_seed(SEED)

CNN_BATCH_SIZE = 64
CNN_EPOCHS = 10
CNN_LR = 1e-3
CNN_L2_REG = 1e-3          # applied via optimizer weight_decay — PyTorch equivalent of Keras kernel_regularizer=l2(...)
CNN_LABEL_SMOOTHING = 0.1
CNN_EARLY_STOPPING_PATIENCE = 3
CNN_LR_PATIENCE = 2

# Class weights for the imbalanced label distribution (mirrors Keras class_weight="balanced")
class_weights_arr = compute_class_weight("balanced", classes=np.unique(y_train), y=y_train)
class_weight_dict = dict(enumerate(class_weights_arr))
print("Class weights:", class_weight_dict)
pos_weight = torch.tensor(class_weights_arr[1] / class_weights_arr[0], dtype=torch.float32).to(device)


def to_tensor_loader(X_seq, y, batch_size, shuffle):
    dataset = TensorDataset(
        torch.tensor(X_seq, dtype=torch.long),
        torch.tensor(y.values if hasattr(y, "values") else y, dtype=torch.float32),
    )
    return DataLoader(dataset, batch_size=batch_size, shuffle=shuffle)


train_loader_cnn = to_tensor_loader(X_tr_seq, y_train, CNN_BATCH_SIZE, shuffle=True)
val_loader_cnn   = to_tensor_loader(X_vl_seq, y_val, CNN_BATCH_SIZE, shuffle=False)
test_loader_cnn  = to_tensor_loader(X_te_seq, y_test, CNN_BATCH_SIZE, shuffle=False)

criterion_cnn = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
optimizer_cnn = torch.optim.Adam(model_cnn2d.parameters(), lr=CNN_LR, weight_decay=CNN_L2_REG)
scheduler_cnn = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer_cnn, mode="min", factor=0.5, patience=CNN_LR_PATIENCE, min_lr=1e-6
)


def smooth_labels(y, smoothing=CNN_LABEL_SMOOTHING):
    return y * (1.0 - smoothing) + 0.5 * smoothing


def run_cnn_epoch(model, loader, optimizer=None):
    is_train = optimizer is not None
    model.train() if is_train else model.eval()

    total_loss, correct, total = 0.0, 0, 0
    with torch.set_grad_enabled(is_train):
        for X_batch, y_batch in loader:
            X_batch, y_batch = X_batch.to(device), y_batch.to(device)
            logits = model(X_batch)
            loss = criterion_cnn(logits, smooth_labels(y_batch))

            if is_train:
                optimizer.zero_grad()
                loss.backward()
                optimizer.step()

            preds = (torch.sigmoid(logits) >= 0.5).float()
            correct += (preds == y_batch).sum().item()
            total += y_batch.size(0)
            total_loss += loss.item() * y_batch.size(0)

    return total_loss / total, correct / total


print("\nInitiating CNN2D training loop...")
start_time = time.time()

history_cnn2d = {"accuracy": [], "loss": [], "val_accuracy": [], "val_loss": []}
best_val_loss = float("inf")
best_state_dict_cnn = None
epochs_without_improvement = 0

for epoch in range(1, CNN_EPOCHS + 1):
    train_loss, train_acc = run_cnn_epoch(model_cnn2d, train_loader_cnn, optimizer_cnn)
    val_loss, val_acc = run_cnn_epoch(model_cnn2d, val_loader_cnn)
    scheduler_cnn.step(val_loss)

    history_cnn2d["loss"].append(train_loss)
    history_cnn2d["accuracy"].append(train_acc)
    history_cnn2d["val_loss"].append(val_loss)
    history_cnn2d["val_accuracy"].append(val_acc)

    print(f"Epoch {epoch}/{CNN_EPOCHS} — loss: {train_loss:.4f} - accuracy: {train_acc:.4f} "
          f"- val_loss: {val_loss:.4f} - val_accuracy: {val_acc:.4f}")

    # Early stopping on val_loss, patience=3, restore_best_weights=True
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        best_state_dict_cnn = copy.deepcopy(model_cnn2d.state_dict())
        epochs_without_improvement = 0
    else:
        epochs_without_improvement += 1
        if epochs_without_improvement >= CNN_EARLY_STOPPING_PATIENCE:
            print(f"  -> Early stopping triggered (no val_loss improvement for {CNN_EARLY_STOPPING_PATIENCE} epochs)")
            break

if best_state_dict_cnn is not None:
    model_cnn2d.load_state_dict(best_state_dict_cnn)
    print(f"\n✔ Restored best weights (val_loss = {best_val_loss:.4f})")

cnn2d_train_time = time.time() - start_time
print(f"\n✔ CNN2D model training completed in {cnn2d_train_time:.1f} seconds")

Class weights: {0: np.float64(1.6684619238476954), 1: np.float64(0.7139564797941902)}

Initiating CNN2D training loop...
Epoch 1/10 — loss: 0.3380 - accuracy: 0.7242 - val_loss: 0.2514 - val_accuracy: 0.8813
Epoch 2/10 — loss: 0.2770 - accuracy: 0.8506 - val_loss: 0.2327 - val_accuracy: 0.8957
Epoch 3/10 — loss: 0.2609 - accuracy: 0.8750 - val_loss: 0.2261 - val_accuracy: 0.9049
Epoch 4/10 — loss: 0.2528 - accuracy: 0.8884 - val_loss: 0.2234 - val_accuracy: 0.9102
Epoch 5/10 — loss: 0.2459 - accuracy: 0.8971 - val_loss: 0.2202 - val_accuracy: 0.9162
Epoch 6/10 — loss: 0.2383 - accuracy: 0.9061 - val_loss: 0.2154 - val_accuracy: 0.9199
Epoch 7/10 — loss: 0.2323 - accuracy: 0.9142 - val_loss: 0.2121 - val_accuracy: 0.9257
Epoch 8/10 — loss: 0.2294 - accuracy: 0.9175 - val_loss: 0.2112 - val_accuracy: 0.9275
Epoch 9/10 — loss: 0.2259 - accuracy: 0.9220 - val_loss: 0.2101 - val_accuracy: 0.9322
Epoch 10/10 — loss: 0.2221 - accuracy: 0.9267 - val_loss: 0.2093 - val_accuracy: 0.9317

✔ Resto

In [ ]:
import json
for name, path in [
    ("v1 (weight=1.0)",  "results/fake_review_detector_v2_consistency_training_w1.0_backup.json"),
    ("v2 (weight=4.0)",  "results/fake_review_detector_v2_consistency_training_w4.0_backup.json"),
    ("v3 (weight=4.0+len aug)", "results/fake_review_detector_v2_consistency_training.json"),
    ("TF-IDF + LogReg",  "results/fake_review_detector_tfidf_training.json"),
]:
    d = json.load(open(path, encoding="utf-8"))
    acc = d["test_metrics"]["accuracy"]
    print(f"{name:28s} test_accuracy={acc:.4f}")
